# Week 2, Day 3: Normalization on the TikTok Comments Dataset, Stored in PostgreSQL

This notebook normalizes the TikTok comments export (800 comments, 9 columns) and loads the result into **PostgreSQL**, using the same connection pattern as the Week 2 Day 1 notebook (`.env` credentials, `DBConnection` class, `with_retry` decorator, logging).

| Part | What happens |
|---|---|
| 1 | Logging, credentials, database connection |
| 2 | Load and clean the CSV, format normalization |
| 3 | Scaling normalization (Min-Max, Z-score, log, Robust) |
| 4 | Database normalization: schema design (1NF to 3NF) and creation in Postgres |
| 5 | Load the normalized data into Postgres |
| 6 | Verify with SQL, test constraints, and redo the scaling **inside** Postgres with window functions |

**Requirements:** `pip install pandas numpy scikit-learn psycopg2-binary python-dotenv`, a running PostgreSQL server, and `tiktok.csv` in the same folder as this notebook.

**`.env` file** (create it next to the notebook, and add `.env` to `.gitignore` so the password is never committed):

```
DB_HOST=127.0.0.1
DB_PORT=5432
DB_NAME=bmc
DB_USER=postgres
DB_PASSWORD=your_password_here
```

In [4]:
import logging
import os
import re
import time
import unicodedata

import numpy as np
import pandas as pd
import psycopg2
from dotenv import load_dotenv
from psycopg2 import OperationalError, Error as Psycopg2Error, sql
from psycopg2.extras import RealDictCursor, execute_values
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 40)

CSV_PATH = "data/tiktok.csv"            
OUT_CSV = "data/tiktok_normalized.csv"  

# Part 1: Logging, credentials, and connection

In [5]:
logging.basicConfig(
    filename="w2day3.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("internship_logger")
logger.info("Week 2 Day 3 - Started")
print("Logging configured -> w2day3.log")

Logging configured -> w2day3.log


In [6]:
load_dotenv(override=True)  # looks for a ".env" file relative to this notebook

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")  # never printed or committed

_required = {"DB_HOST": DB_HOST, "DB_PORT": DB_PORT, "DB_NAME": DB_NAME,
             "DB_USER": DB_USER, "DB_PASSWORD": DB_PASSWORD}
_missing = [k for k, v in _required.items() if not v]
if _missing:
    raise RuntimeError(
        f"Missing required .env variable(s): {', '.join(_missing)}. "
        f"Please check your .env file."
    )
logger.info("Credentials loaded from .env")
print("Credentials loaded from .env")

Credentials loaded from .env


### Create the database if it does not exist
`CREATE DATABASE` cannot run inside a transaction and cannot be run while connected to the database you are creating, so we connect to the default `postgres` database with autocommit on. The role needs the `CREATEDB` privilege; if it does not have it, the step logs a warning and the notebook continues (the database may already exist).

In [7]:
def ensure_database(host, port, dbname, user, password):
    """Create `dbname` if it is missing. Returns True if it was created."""
    try:
        admin = psycopg2.connect(host=host, port=port, dbname="postgres",
                                 user=user, password=password)
    except OperationalError as e:
        logger.warning("Could not check/create database '%s': %s", dbname, e)
        print("Skipped database check (could not connect to the 'postgres' database)")
        return False
    admin.autocommit = True
    try:
        with admin.cursor() as cur:
            cur.execute("SELECT 1 FROM pg_database WHERE datname = %s;", (dbname,))
            if cur.fetchone():
                logger.info("Database '%s' already exists", dbname)
                print(f"Database '{dbname}' already exists")
                return False
            cur.execute(sql.SQL("CREATE DATABASE {};").format(sql.Identifier(dbname)))
            logger.info("Database '%s' created", dbname)
            print(f"Database '{dbname}' created")
            return True
    except Psycopg2Error as e:
        logger.warning("Could not create database '%s': %s", dbname, e)
        print(f"Could not create database '{dbname}': {e.pgerror or e}")
        return False
    finally:
        admin.close()


_ = ensure_database(DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD)

Database 'BMC' already exists


In [8]:
class DBConnection:
    """Small wrapper so every part of the pipeline shares one connection."""

    def __init__(self, host, port, dbname, user, password):
        self.host, self.port, self.dbname, self.user, self.password = (
            host, port, dbname, user, password
        )
        self._connect()

    def _connect(self):
        self.conn = psycopg2.connect(
            host=self.host, port=self.port, dbname=self.dbname,
            user=self.user, password=self.password,
        )
        logger.info("Connected to database '%s'", self.dbname)

    def reconnect(self):
        """Used by the retry decorator when the connection itself has died."""
        try:
            self.conn.close()
        except Exception:
            pass
        self._connect()
        logger.info("Reconnected to database '%s'", self.dbname)

    def cursor(self):
        return self.conn.cursor(cursor_factory=RealDictCursor)

    def commit(self):
        self.conn.commit()

    def rollback(self):
        self.conn.rollback()

    def close(self):
        self.conn.close()
        logger.info("Connection closed")


def with_retry(max_attempts=3, delay_seconds=1):
    """Retries a db-operation function up to max_attempts times on
    psycopg2.OperationalError, always rolling back (or reconnecting)
    before each retry so the connection never sits in a broken state.
    The decorated function's first positional argument must be the db object.
    """
    def decorator(func):
        def wrapper(db, *args, **kwargs):
            attempt = 0
            while True:
                attempt += 1
                try:
                    return func(db, *args, **kwargs)
                except OperationalError as e:
                    logger.error(
                        "Attempt %d/%d for %s failed: %s",
                        attempt, max_attempts, func.__name__, e,
                    )
                    try:
                        db.rollback()
                    except Exception:
                        # Connection itself is dead (not just the transaction) - reconnect.
                        try:
                            db.reconnect()
                        except Exception as reconnect_err:
                            logger.error("Reconnect failed: %s", reconnect_err)
                    if attempt >= max_attempts:
                        logger.error(
                            "Giving up on %s after %d attempts", func.__name__, max_attempts
                        )
                        raise
                    time.sleep(delay_seconds)
        return wrapper
    return decorator


db = DBConnection(host=DB_HOST, port=DB_PORT, dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD)
print("Connected")

Connected


# Part 2: Load, clean, and format-normalize the CSV

### 2.1 Load IDs as strings
TikTok IDs have 19 digits. A `float64` column is exact only up to 2^53 (16 digits), so an ID column with blanks that pandas auto-reads as float is **silently corrupted**. We read the three ID columns as strings and prove why.

In [9]:
naive = pd.read_csv(CSV_PATH)
print("Default dtypes:")
print(naive[["Comment ID", "Reply to Which Comment", "User ID"]].dtypes)

df_raw = pd.read_csv(
    CSV_PATH,
    dtype={"Comment ID": "string", "Reply to Which Comment": "string", "User ID": "string"},
)

valid_ids = set(df_raw["Comment ID"])
naive_parents = naive["Reply to Which Comment"].dropna().astype("int64").astype(str)
good_parents = df_raw["Reply to Which Comment"].dropna()

print()
print(f"Parent links matching a Comment ID, default float read : {naive_parents.isin(valid_ids).sum()} / {len(naive_parents)}")
print(f"Parent links matching a Comment ID, string read        : {good_parents.isin(valid_ids).sum()} / {len(good_parents)}")
print("Shape:", df_raw.shape)
logger.info("Loaded %s: %d rows, %d columns", CSV_PATH, *df_raw.shape)

Default dtypes:
Comment ID                  int64
Reply to Which Comment    float64
User ID                     int64
dtype: object

Parent links matching a Comment ID, default float read : 0 / 369
Parent links matching a Comment ID, string read        : 369 / 369
Shape: (800, 9)


### 2.2 Column names and structural missing values
`reply_count` and `is_pinned` are blank on **exactly the reply rows**: a reply cannot have replies or be pinned. These are structural blanks, not errors, so we fill them with 0 / "No" (Week 1 Day 3: find out why values are missing before filling).

In [10]:
df = df_raw.copy()
df.columns = (
    df.columns.str.strip().str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")
)
df = df.rename(columns={
    "reply_to_which_comment": "parent_comment_id",
    "comment": "comment_text",
    "comment_time": "created_at",
    "pinned_to_top": "is_pinned",
})

df["is_reply"] = df["parent_comment_id"].notna()
print("Is reply_count missing exactly on reply rows?")
print(pd.crosstab(df["is_reply"], df["reply_count"].isna(), rownames=["is_reply"], colnames=["reply_count missing"]))

df["reply_count"] = df["reply_count"].fillna(0).astype(int)
df["is_pinned"] = df["is_pinned"].fillna("No")
print("\nRemaining blanks in reply_count / is_pinned:", int(df[["reply_count", "is_pinned"]].isna().sum().sum()))

Is reply_count missing exactly on reply rows?
reply_count missing  False  True 
is_reply                         
False                  431      0
True                     0    369

Remaining blanks in reply_count / is_pinned: 0


### 2.3 Format normalization
One representation per fact: real timestamps, 1/0 flags, and normalized text (Unicode NFKC, lowercase, collapsed whitespace). Emoji are kept because they are the signal in this dataset.

In [11]:
df["created_at"] = pd.to_datetime(df["created_at"], format="%m/%d/%Y, %I:%M:%S %p")
for col in ["author_digged", "is_pinned"]:
    df[col] = df[col].map({"Yes": 1, "No": 0}).astype(int)


def normalize_text(s):
    if pd.isna(s):
        return ""
    s = unicodedata.normalize("NFKC", s)
    s = s.lower()
    return re.sub(r"\s+", " ", s).strip()


df["has_text"] = df["comment_text"].notna() & (df["comment_text"].str.strip() != "")
df["comment_clean"] = df["comment_text"].map(normalize_text)

df["comment_length"] = df["comment_clean"].str.len()
df["word_count"] = df["comment_clean"].str.split().str.len()
df["hour"] = df["created_at"].dt.hour
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)   # cyclical encoding: 23:00 is next to 00:00
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

changed = df["comment_text"].fillna("") != df["comment_clean"]
print(f"Rows changed by text normalization: {changed.sum()} of {len(df)}")
print(f"Rows with no text: {(~df['has_text']).sum()}")
print("Date range:", df["created_at"].min(), "->", df["created_at"].max())
print("is_pinned unique values:", df["is_pinned"].unique(), "(constant column)")
logger.info("Cleaning and format normalization complete")

Rows changed by text normalization: 339 of 800
Rows with no text: 12
Date range: 2026-07-22 21:42:58 -> 2026-08-06 12:55:05
is_pinned unique values: [0] (constant column)


# Part 3: Scaling normalization

### 3.1 Distributions first
The distribution decides the method. The count columns are heavily skewed: most comments have 0 likes.

In [12]:
num_cols = ["digg_count", "reply_count", "comment_length", "word_count"]

stats = df[num_cols].agg(["min", "max", "mean", "median", "std", "skew"]).T
stats["iqr"] = df[num_cols].quantile(0.75) - df[num_cols].quantile(0.25)
print(stats.round(3))
print()
print("digg_count value counts:")
print(df["digg_count"].value_counts().sort_index().to_string())

                min    max    mean  median     std   skew   iqr
digg_count      0.0   10.0   0.238     0.0   0.701  6.511   0.0
reply_count     0.0   12.0   0.461     0.0   1.129  4.441   1.0
comment_length  0.0  274.0  43.745    38.0  32.925  1.897  35.0
word_count      0.0   51.0   8.158     7.0   6.023  1.880   7.0

digg_count value counts:
digg_count
0     661
1     114
2      15
3       6
5       1
6       1
7       1
10      1


### 3.2 Min-Max, Z-score, log transform, Robust

| Method | Formula | Result | Use when |
|---|---|---|---|
| Min-Max | (x - min) / (max - min) | 0 to 1 | bounded features, no big outliers |
| Z-score | (x - mean) / std | mean 0, std 1 | roughly bell-shaped data |
| log1p + Min-Max | min-max of log(1 + x) | 0 to 1 | skewed counts with zeros |
| Robust | (x - median) / IQR | median 0 | outliers, and IQR must not be 0 |

In [13]:
def min_max(s):
    return (s - s.min()) / (s.max() - s.min())

def z_score(s):
    return (s - s.mean()) / s.std(ddof=0)   # ddof=0 matches scikit-learn and SQL STDDEV_POP

scaled = pd.DataFrame(index=df.index)
for c in num_cols:
    scaled[f"{c}_minmax"] = min_max(df[c])
    scaled[f"{c}_z"] = z_score(df[c])
for c in ["digg_count", "reply_count"]:
    scaled[f"{c}_log_minmax"] = min_max(np.log1p(df[c]))

q1, q3 = df["comment_length"].quantile([0.25, 0.75])
scaled["comment_length_robust"] = (df["comment_length"] - df["comment_length"].median()) / (q3 - q1)

# Verify every manual result against scikit-learn
print("Min-Max  == sklearn:", np.allclose(MinMaxScaler().fit_transform(df[num_cols]),
                                          scaled[[f"{c}_minmax" for c in num_cols]]))
print("Z-score  == sklearn:", np.allclose(StandardScaler().fit_transform(df[num_cols]),
                                          scaled[[f"{c}_z" for c in num_cols]]))
print("Robust   == sklearn:", np.allclose(RobustScaler().fit_transform(df[["comment_length"]]).ravel(),
                                          scaled["comment_length_robust"]))
print()
print(scaled.agg(["min", "max", "mean", "std"]).T.round(3))
logger.info("Scaling normalization complete (%d scaled columns)", scaled.shape[1])

Min-Max  == sklearn: True
Z-score  == sklearn: True
Robust   == sklearn: True

                          min     max   mean    std
digg_count_minmax       0.000   1.000  0.024  0.070
digg_count_z           -0.339  13.931  0.000  1.001
reply_count_minmax      0.000   1.000  0.038  0.094
reply_count_z          -0.409  10.225  0.000  1.001
comment_length_minmax   0.000   1.000  0.160  0.120
comment_length_z       -1.329   6.998  0.000  1.001
word_count_minmax       0.000   1.000  0.160  0.118
word_count_z           -1.355   7.118 -0.000  1.001
digg_count_log_minmax   0.000   1.000  0.058  0.137
reply_count_log_minmax  0.000   1.000  0.093  0.178
comment_length_robust  -1.086   6.743  0.164  0.941


In [14]:
# Why log1p for digg_count: same comment, three scalings
cmp = (pd.concat([df[["digg_count"]], scaled[["digg_count_minmax", "digg_count_z", "digg_count_log_minmax"]]], axis=1)
       .drop_duplicates("digg_count").sort_values("digg_count"))
print(cmp.round(3).to_string(index=False))

# Leakage: fit on train only, then transform test
train, test = train_test_split(df[["comment_length"]], test_size=0.2, random_state=42)
scaler = MinMaxScaler().fit(train)
print()
print(f"train min/max after scaling: {scaler.transform(train).min():.3f} / {scaler.transform(train).max():.3f}")
print(f"test  min/max after scaling: {scaler.transform(test).min():.3f} / {scaler.transform(test).max():.3f}")
print(f"A new 300-character comment scales to {scaler.transform(pd.DataFrame({'comment_length': [300]}))[0][0]:.3f} (training max was {scaler.data_max_[0]:.0f})")

 digg_count  digg_count_minmax  digg_count_z  digg_count_log_minmax
          0                0.0        -0.339                  0.000
          1                0.1         1.088                  0.289
          2                0.2         2.515                  0.458
          3                0.3         3.942                  0.578
          5                0.5         6.796                  0.747
          6                0.6         8.223                  0.812
          7                0.7         9.650                  0.867
         10                1.0        13.931                  1.000

train min/max after scaling: 0.000 / 1.000
test  min/max after scaling: 0.000 / 0.719
A new 300-character comment scales to 1.095 (training max was 274)


# Part 4: Database normalization and schema creation in Postgres

### 4.1 Normal-form check on the flat CSV

| Normal form | Question | This dataset |
|---|---|---|
| 1NF | Is every cell a single atomic value? | Yes |
| 2NF | Does every column depend on the whole key? | Yes, the key is one column (`comment_id`) |
| 3NF | Is any column derivable from other data? | **`reply_count` is derived** from `parent_comment_id`; `user_id` repeats across rows |

In [15]:
top_level = df[~df["is_reply"]].copy()
derived = df["parent_comment_id"].value_counts()
top_level["derived_reply_count"] = top_level["comment_id"].map(derived).fillna(0).astype(int)

match = (top_level["derived_reply_count"] == top_level["reply_count"]).sum()
print(f"Stored reply_count equals the number of reply rows for {match} of {len(top_level)} top-level comments")
print("All replies point to top-level comments:", df.loc[df["is_reply"], "parent_comment_id"].isin(top_level["comment_id"]).all())
print(f"Rows: {len(df)}, distinct users: {df['user_id'].nunique()}")

# Postgres BIGINT holds up to 2^63 - 1; make sure every ID fits
max_id = max(df["comment_id"].astype("int64").max(), df["user_id"].astype("int64").max())
print(f"Largest ID: {max_id} (BIGINT limit: {2**63 - 1})  fits: {max_id < 2**63}")

Stored reply_count equals the number of reply rows for 431 of 431 top-level comments
All replies point to top-level comments: True
Rows: 800, distinct users: 435
Largest ID: 7670812229898683137 (BIGINT limit: 9223372036854775807)  fits: True


### 4.2 Target schema

| Table | Key | Purpose |
|---|---|---|
| `users` | `user_id` (PK) | one row per distinct user |
| `comments` | `comment_id` (PK) | the source of truth: FK to `users`, self-referencing FK `parent_comment_id`, CHECK constraints |
| `comment_stats` (view) | | `reply_count` computed on demand instead of stored |
| `comment_features` | `comment_id` (PK, FK) | derived scaled features for ML, kept separate so it can be rebuilt any time |

Type choices: `BIGINT` for 19-digit IDs, `TIMESTAMP` for time, `BOOLEAN` for the Yes/No flags, `DOUBLE PRECISION` for scaled values. Postgres does not index foreign keys automatically, so we add indexes on `user_id` and `parent_comment_id` (both are used in joins).

In [16]:
DDL_TABLES = """
CREATE TABLE IF NOT EXISTS users (
    user_id BIGINT PRIMARY KEY
);

CREATE TABLE IF NOT EXISTS comments (
    comment_id        BIGINT PRIMARY KEY,
    user_id           BIGINT NOT NULL REFERENCES users(user_id),
    parent_comment_id BIGINT REFERENCES comments(comment_id),
    comment_text      TEXT,
    created_at        TIMESTAMP NOT NULL,
    digg_count        INTEGER NOT NULL CHECK (digg_count >= 0),
    author_digged     BOOLEAN NOT NULL,
    is_pinned         BOOLEAN NOT NULL
);

CREATE INDEX IF NOT EXISTS idx_comments_user   ON comments(user_id);
CREATE INDEX IF NOT EXISTS idx_comments_parent ON comments(parent_comment_id);

CREATE TABLE IF NOT EXISTS comment_features (
    comment_id             BIGINT PRIMARY KEY REFERENCES comments(comment_id),
    comment_length         INTEGER NOT NULL,
    word_count             INTEGER NOT NULL,
    hour                   INTEGER NOT NULL CHECK (hour BETWEEN 0 AND 23),
    hour_sin               DOUBLE PRECISION NOT NULL,
    hour_cos               DOUBLE PRECISION NOT NULL,
    digg_count_minmax      DOUBLE PRECISION NOT NULL,
    digg_count_z           DOUBLE PRECISION NOT NULL,
    digg_count_log_minmax  DOUBLE PRECISION NOT NULL,
    reply_count_minmax     DOUBLE PRECISION NOT NULL,
    reply_count_z          DOUBLE PRECISION NOT NULL,
    reply_count_log_minmax DOUBLE PRECISION NOT NULL,
    comment_length_minmax  DOUBLE PRECISION NOT NULL,
    comment_length_z       DOUBLE PRECISION NOT NULL,
    comment_length_robust  DOUBLE PRECISION NOT NULL,
    word_count_minmax      DOUBLE PRECISION NOT NULL,
    word_count_z           DOUBLE PRECISION NOT NULL
);

CREATE OR REPLACE VIEW comment_stats AS
SELECT c.comment_id, c.user_id, c.digg_count,
       COUNT(r.comment_id) AS reply_count
FROM comments c
LEFT JOIN comments r ON r.parent_comment_id = c.comment_id
WHERE c.parent_comment_id IS NULL
GROUP BY c.comment_id, c.user_id, c.digg_count;
"""


@with_retry()
def create_schema(db):
    try:
        with db.cursor() as cur:
            cur.execute(DDL_TABLES)
        db.commit()
        logger.info("Schema created/verified (users, comments, comment_features, comment_stats)")
        print("Schema created/verified")
    except Psycopg2Error as e:
        db.rollback()
        logger.error("Schema creation failed: %s", e)
        raise


create_schema(db)

Schema created/verified


# Part 5: Load the normalized data

Order matters because of the foreign keys: `users` first, then top-level comments, then replies (a reply needs its parent to exist), then features. The whole load, including the `TRUNCATE` that makes the notebook re-runnable, is **one transaction**: it either fully succeeds or leaves the tables untouched, which also makes it safe for `with_retry` to run again.

In [17]:
# Convert pandas/numpy values into plain Python values that psycopg2 can adapt
users_rows = [(int(u),) for u in df["user_id"].drop_duplicates()]

def comment_row(r):
    return (
        int(r.comment_id),
        int(r.user_id),
        int(r.parent_comment_id) if pd.notna(r.parent_comment_id) else None,
        r.comment_text if pd.notna(r.comment_text) else None,
        r.created_at.to_pydatetime(),
        int(r.digg_count),
        bool(r.author_digged),
        bool(r.is_pinned),
    )

ordered = df.sort_values("is_reply", kind="stable")            # top-level first, replies after
top_rows = [comment_row(r) for r in ordered[~ordered["is_reply"]].itertuples()]
reply_rows = [comment_row(r) for r in ordered[ordered["is_reply"]].itertuples()]

feature_cols = ["comment_length", "word_count", "hour", "hour_sin", "hour_cos"]
features = pd.concat([df[["comment_id"] + feature_cols], scaled], axis=1)
db_feature_order = [
    "comment_length", "word_count", "hour", "hour_sin", "hour_cos",
    "digg_count_minmax", "digg_count_z", "digg_count_log_minmax",
    "reply_count_minmax", "reply_count_z", "reply_count_log_minmax",
    "comment_length_minmax", "comment_length_z", "comment_length_robust",
    "word_count_minmax", "word_count_z",
]
feature_rows = [
    (int(cid), *vals)
    for cid, vals in zip(features["comment_id"], features[db_feature_order].astype(object).values.tolist())
]

print(f"users: {len(users_rows)} | top-level comments: {len(top_rows)} | replies: {len(reply_rows)} | feature rows: {len(feature_rows)}")

users: 435 | top-level comments: 431 | replies: 369 | feature rows: 800


In [18]:
INSERT_USERS = "INSERT INTO users (user_id) VALUES %s;"
INSERT_COMMENTS = """
INSERT INTO comments (comment_id, user_id, parent_comment_id, comment_text,
                      created_at, digg_count, author_digged, is_pinned)
VALUES %s;
"""
INSERT_FEATURES = "INSERT INTO comment_features (comment_id, " + ", ".join(db_feature_order) + ") VALUES %s;"


@with_retry()
def load_all(db):
    try:
        with db.cursor() as cur:
            cur.execute("TRUNCATE TABLE comment_features, comments, users;")
            logger.info("Truncated comment_features, comments, users before reloading")
            execute_values(cur, INSERT_USERS, users_rows)
            execute_values(cur, INSERT_COMMENTS, top_rows)
            execute_values(cur, INSERT_COMMENTS, reply_rows)
            execute_values(cur, INSERT_FEATURES, feature_rows)
        db.commit()
        logger.info("Loaded %d users, %d comments, %d feature rows",
                    len(users_rows), len(top_rows) + len(reply_rows), len(feature_rows))
        print(f"Loaded {len(users_rows)} users, {len(top_rows) + len(reply_rows)} comments, {len(feature_rows)} feature rows")
    except Psycopg2Error as e:
        db.rollback()
        logger.error("Data load failed: %s", e)
        raise


load_all(db)

Loaded 435 users, 800 comments, 800 feature rows


# Part 6: Verify in Postgres

### 6.1 Schema and row counts

In [19]:
@with_retry()
def query_df(db, query, params=None):
    """Run a SELECT and return the result as a DataFrame."""
    with db.cursor() as cur:
        cur.execute(query, params)
        rows = cur.fetchall()
    return pd.DataFrame(rows)


print(query_df(db, """
    SELECT table_name, column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_schema = 'public' AND table_name IN ('users', 'comments')
    ORDER BY table_name, ordinal_position;
""").to_string(index=False))

print()
print(query_df(db, """
    SELECT 'users' AS table_name, COUNT(*) AS n FROM users
    UNION ALL SELECT 'comments', COUNT(*) FROM comments
    UNION ALL SELECT 'comment_features', COUNT(*) FROM comment_features;
""").to_string(index=False))

table_name       column_name                   data_type is_nullable
  comments        comment_id                      bigint          NO
  comments           user_id                      bigint          NO
  comments parent_comment_id                      bigint         YES
  comments      comment_text                        text         YES
  comments        created_at timestamp without time zone          NO
  comments        digg_count                     integer          NO
  comments     author_digged                     boolean          NO
  comments         is_pinned                     boolean          NO
     users           user_id                      bigint          NO

      table_name   n
           users 435
        comments 800
comment_features 800


### 6.2 No information was lost

In [20]:
@with_retry()
def verify(db):
    checks = {}

    # 1) The reply_count VIEW equals the original stored column
    view = query_df(db, "SELECT comment_id, reply_count FROM comment_stats;")
    orig = top_level[["comment_id", "reply_count"]].assign(comment_id=lambda x: x["comment_id"].astype("int64"))
    merged = view.merge(orig, on="comment_id", suffixes=("_view", "_original"))
    checks["view reply_count == original (all top-level)"] = bool((merged["reply_count_view"] == merged["reply_count_original"]).all()) and len(merged) == len(top_level)

    # 2) A JOIN rebuilds the flat row count
    n_join = query_df(db, "SELECT COUNT(*) AS n FROM comments c JOIN users u ON u.user_id = c.user_id;").iloc[0]["n"]
    checks["JOIN row count == CSV row count"] = int(n_join) == len(df)

    # 3) Comment IDs round-trip exactly (no float corruption)
    stored = set(query_df(db, "SELECT comment_id FROM comments;")["comment_id"])
    checks["all 800 comment IDs stored exactly"] = stored == set(df["comment_id"].astype("int64"))

    # 4) Every reply points to a stored comment
    orphans = query_df(db, """
        SELECT COUNT(*) AS n FROM comments r
        LEFT JOIN comments p ON p.comment_id = r.parent_comment_id
        WHERE r.parent_comment_id IS NOT NULL AND p.comment_id IS NULL;
    """).iloc[0]["n"]
    checks["orphan replies == 0"] = int(orphans) == 0

    for name, ok in checks.items():
        print(f"{'PASS' if ok else 'FAIL'}  {name}")
    logger.info("Verification: %s", checks)
    return checks


_ = verify(db)

PASS  view reply_count == original (all top-level)
PASS  JOIN row count == CSV row count
PASS  all 800 comment IDs stored exactly
PASS  orphan replies == 0


In [21]:
print("Top 5 most active users:")
print(query_df(db, """
    SELECT user_id, COUNT(*) AS comments_written
    FROM comments GROUP BY user_id
    ORDER BY comments_written DESC, user_id LIMIT 5;
""").to_string(index=False))

print()
print("Top 5 comments by replies:")
print(query_df(db, """
    SELECT s.comment_id, s.reply_count, s.digg_count,
           COALESCE(LEFT(c.comment_text, 45), '(no text)') AS comment_text
    FROM comment_stats s JOIN comments c ON c.comment_id = s.comment_id
    ORDER BY s.reply_count DESC, s.comment_id LIMIT 5;
""").to_string(index=False))

Top 5 most active users:
            user_id  comments_written
7588871257367266322               196
6887148164795122694                 6
6949018073941869570                 6
7568861125161698324                 6
6522287339984699394                 5

Top 5 comments by replies:
         comment_id  reply_count  digg_count                                  comment_text
7666685642684138247           12           5 sir biometric ko form khulyo ki naii date nai
7668938470476546836            9           6                               yesto vanxa sir
7670373282600928021            9           0                                     (no text)
7666379879197852423            8           2       Medical ko option ma aaywsi K Garne ho?
7668113312652264209            8           0            maro yasto xa k garne hora aba sir


### 6.3 Constraints protect the data
A failed statement puts the transaction in an aborted state, so each test ends with `rollback()`.

In [22]:
tests = {
    "foreign key (unknown user)": (
        "INSERT INTO comments (comment_id, user_id, created_at, digg_count, author_digged, is_pinned) "
        "VALUES (1, 999, NOW(), 0, FALSE, FALSE);"
    ),
    "check (negative digg_count)": (
        "INSERT INTO comments (comment_id, user_id, created_at, digg_count, author_digged, is_pinned) "
        "SELECT 2, user_id, NOW(), -5, FALSE, FALSE FROM users LIMIT 1;"
    ),
    "primary key (duplicate comment_id)": (
        "INSERT INTO comments (comment_id, user_id, created_at, digg_count, author_digged, is_pinned) "
        "SELECT comment_id, user_id, NOW(), 0, FALSE, FALSE FROM comments LIMIT 1;"
    ),
}

for name, stmt in tests.items():
    try:
        with db.cursor() as cur:
            cur.execute(stmt)
        db.commit()
        print(f"NOT REJECTED: {name}")
    except Psycopg2Error as e:
        db.rollback()
        print(f"Rejected  {name:<38} -> {type(e).__name__}")
        logger.info("Constraint test '%s' rejected as expected: %s", name, type(e).__name__)

Rejected  foreign key (unknown user)             -> ForeignKeyViolation
Rejected  check (negative digg_count)            -> CheckViolation
Rejected  primary key (duplicate comment_id)     -> UniqueViolation


### 6.4 Redo the scaling inside Postgres (window functions)
In a data pipeline the scaling often happens in SQL, close to the data. `MIN() OVER ()`, `MAX() OVER ()`, `AVG() OVER ()` and `STDDEV_POP() OVER ()` compute the whole-column statistics per row. We compare the SQL results with the pandas results stored in `comment_features`.

In [23]:
sql_scaled = query_df(db, """
    SELECT comment_id,
           ((digg_count - MIN(digg_count) OVER ())::float8
             / NULLIF(MAX(digg_count) OVER () - MIN(digg_count) OVER (), 0)) AS digg_minmax_sql,
           ((digg_count - AVG(digg_count) OVER ())
             / NULLIF(STDDEV_POP(digg_count) OVER (), 0))::float8            AS digg_z_sql
    FROM comments;
""")
stored = query_df(db, "SELECT comment_id, digg_count_minmax, digg_count_z FROM comment_features;")

both = sql_scaled.merge(stored, on="comment_id")
print("Rows compared:", len(both))
print("SQL Min-Max == pandas Min-Max:", np.allclose(both["digg_minmax_sql"], both["digg_count_minmax"]))
print("SQL Z-score == pandas Z-score:", np.allclose(both["digg_z_sql"], both["digg_count_z"]))

print()
print("Range check on stored features:")
print(query_df(db, """
    SELECT MIN(digg_count_minmax)      AS digg_min,  MAX(digg_count_minmax)      AS digg_max,
           MIN(comment_length_minmax)  AS len_min,   MAX(comment_length_minmax)  AS len_max,
           ROUND(AVG(comment_length_z)::numeric, 6)  AS len_z_mean,
           ROUND(STDDEV_POP(comment_length_z)::numeric, 6) AS len_z_std
    FROM comment_features;
""").to_string(index=False))

Rows compared: 800
SQL Min-Max == pandas Min-Max: True
SQL Z-score == pandas Z-score: True

Range check on stored features:
 digg_min  digg_max  len_min  len_max len_z_mean len_z_std
      0.0       1.0      0.0      1.0   0.000000  1.000000


# Part 7: Save and close

In [24]:
final = pd.concat([
    df[["comment_id", "parent_comment_id", "user_id", "comment_text", "comment_clean",
        "created_at", "is_reply", "has_text", "author_digged", "is_pinned"]],
    df[num_cols + ["hour", "hour_sin", "hour_cos"]],
    scaled,
], axis=1)
final.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV, final.shape)

db.close()
logger.info("W2D3 completed successfully")
print("Connection closed. Log written to w2day3.log")

Saved: data/tiktok_normalized.csv (800, 28)
Connection closed. Log written to w2day3.log


## Summary

| Layer | What we did | Takeaway |
|---|---|---|
| Connection | `.env` credentials, `DBConnection`, `with_retry`, logging, auto-create database | Same pattern as Week 2 Day 1, plus an atomic, re-runnable load |
| Cleaning | IDs as strings, structural blanks filled, timestamps and flags normalized | 19-digit IDs break as floats; find out why values are missing |
| Scaling | Min-Max, Z-score, log1p + Min-Max, Robust, verified against scikit-learn | Choose the method from the distribution; fit on train only |
| Database | `users`, `comments`, `comment_features`, `comment_stats` view with PK/FK/CHECK and indexes | Derived data (`reply_count`) is computed, not stored; BIGINT/BOOLEAN/TIMESTAMP fit the data |
| SQL | Window-function Min-Max and Z-score match pandas | Scaling can run inside the database |

